# HGCLR Global ISO Hard-Sibling-Negative — 5-Fold CV, Seed 42

This notebook follows the same evaluation structure as the two-head global CV notebook: fixed 5-fold L2-stratified CV, rotating validation fold, seed 42, out-of-fold predictions, path/routing decoders, hierarchical metrics, error decomposition, per-class/per-parent reports, confusion matrices, and loss plots.

The training architecture remains HGCLR-specific and is not changed:

- RoBERTa L2 classifier
- L1 auxiliary head
- contrastive projection head
- same-L2 positives
- same-parent different-L2 hard sibling negatives


In [ ]:

# ============================================================
# 0) Imports and Drive mount
# ============================================================

import os
import re
import gc
import json
import math
import time
import shutil
import random
import inspect
import warnings
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed,
)


# Colab Drive mount. If not using Colab, comment this block.
try:
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive mount skipped:", e)

print("Torch:", torch.__version__)
import transformers
print("Transformers:", transformers.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

# ============================================================
# 1) Configuration: HGCLR parameters + shared CV/reporting parameters
# ============================================================

CV_DATASET_PATH = "/data/dataset_with_test_fold.xlsx"

TEXT_COL = "text"
L1_COL = "L1"
L2_COL = "L2"

OUTPUT_DIR = "/results/03_iso_hgclr"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "roberta-base"
MAX_EPOCHS = 30
MAX_LEN = 128

BATCH_SIZE = 32
LR = 1e-5
WD = 0.03
WARMUP_RATIO = 0.08

MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 3
SAVE_TOTAL_LIMIT = 1

# ============================================================
# Dropout regularization
# ============================================================
ENCODER_DROPOUT = 0.15        # RoBERTa hidden-layer dropout
ATTENTION_DROPOUT = 0.10      # RoBERTa attention dropout
CLASSIFIER_DROPOUT = 0.25     # L2 classifier + L1 head dropout
PROJECTION_DROPOUT = 0.10     # contrastive projection head dropout

FOCAL_GAMMA = 1.5
MAX_CLASS_WEIGHT_CLIP = 7.5

SEED = 42
SEEDS = [42]  # kept only for compatibility
N_FOLDS = 5

# Keep validation fold because this notebook uses early stopping and best-checkpoint selection.
USE_ROTATING_VALIDATION_FOLD = True

# ============================================================
# HGCLR-style settings - keep current hard sibling negative design
# ============================================================
HGCLR_CONFIGS = [
    {
        "experiment_name": "HGCLR_l1_005_contrast_01_t010_sibneg125",
        "lambda_l1": 0.00,
        "lambda_hgclr": 0.05,
        "contrast_temp": 0.10,
        "same_l2_weight": 1.00,
        "sibling_negative_weight": 1.50,
        "projection_dim": 256,
    },
]

# ============================================================
# Path/routing decoder settings - inference only, same trained model
# ============================================================
DECODER_CONFIGS = [
    {"decoder": "base_l2", "top_k": None, "alpha": None},
    {"decoder": "hard_mask", "top_k": 1, "alpha": None},
    {"decoder": "soft_fusion", "top_k": None, "alpha": 1.00},
    {"decoder": "topk_parent", "top_k": 2, "alpha": None},
    {"decoder": "topk_parent", "top_k": 3, "alpha": None},
]

RUN_HGCLR_CV = True

print("CV dataset:", CV_DATASET_PATH)
print("Output:", OUTPUT_DIR)
print("Seed:", SEED)
print("N_FOLDS:", N_FOLDS)


In [ ]:

# ============================================================
# 2) Load the prepared dataset with fixed fold assignments
# ============================================================

def read_table(path: str) -> pd.DataFrame:
    if path.endswith(".xlsx") or path.endswith(".xls"):
        return pd.read_excel(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

def load_prepared_cv_dataset(path: str, n_folds: int = 5) -> pd.DataFrame:
    """Load the prepared dataset unchanged and validate its saved folds."""
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = read_table(path).copy()

    required = ["id", "orig_id", "source_split", "cv_index", TEXT_COL, L1_COL, L2_COL, "test_fold"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Prepared CV dataset is missing required columns: {missing}. "
            "Use full_dataset_with_test_fold.xlsx from the preparation notebook."
        )
    if df[required].isna().any().any():
        bad_columns = df[required].columns[df[required].isna().any()].tolist()
        raise ValueError(f"Prepared CV dataset contains missing values in: {bad_columns}")

    df["cv_index"] = pd.to_numeric(df["cv_index"], errors="raise").astype(int)
    df["test_fold"] = pd.to_numeric(df["test_fold"], errors="raise").astype(int)
    if df["cv_index"].duplicated().any():
        raise ValueError("cv_index is not unique in the prepared CV dataset.")
    if df["id"].astype(str).duplicated().any():
        raise ValueError("id is not unique in the prepared CV dataset.")

    expected_folds = set(range(n_folds))
    observed_folds = set(df["test_fold"].unique().tolist())
    if observed_folds != expected_folds:
        raise ValueError(
            f"Expected saved folds {sorted(expected_folds)}, found {sorted(observed_folds)}."
        )

    # Compatibility aliases used by the existing HGCLR reporting code.
    df["global_id"] = df["cv_index"]
    df["original_split"] = df["source_split"]

    # Do not repeat merging, cleaning, normalization, or deduplication.
    return df.sort_values("cv_index").reset_index(drop=True)

full_df = load_prepared_cv_dataset(CV_DATASET_PATH, n_folds=N_FOLDS)

print("Loaded full_df:", full_df.shape)
print("Full L1:", full_df[L1_COL].nunique())
print("Full L2:", full_df[L2_COL].nunique())
print("Saved test-fold sizes:")
display(full_df["test_fold"].value_counts().sort_index().rename("n").to_frame())
display(full_df[[TEXT_COL, L1_COL, L2_COL, "original_split", "global_id"]].head())


In [ ]:

# ============================================================
# 3) Label maps, hierarchy maps, and fixed 5-fold assignment
# ============================================================

def build_label_maps(labels: List[str]):
    labs = sorted(pd.Series(labels).astype(str).str.strip().unique().tolist())
    lab2id = {lab: i for i, lab in enumerate(labs)}
    id2lab = {i: lab for lab, i in lab2id.items()}
    return lab2id, id2lab

def encode_labels(df: pd.DataFrame, col: str, lab2id: Dict[str, int]):
    vals = df[col].astype(str).str.strip().tolist()
    missing = sorted(set([v for v in vals if v not in lab2id]))
    if missing:
        raise ValueError(f"Found {col} labels not in map: {missing}")
    return np.array([lab2id[v] for v in vals], dtype=np.int64)

# For ISO classification, the label taxonomy is known. Build output maps from the full combined split.
l2_lab2id, l2_id2lab = build_label_maps(full_df[L2_COL].tolist())
l1_lab2id, l1_id2lab = build_label_maps(full_df[L1_COL].tolist())

num_l2 = len(l2_lab2id)
num_l1 = len(l1_lab2id)

y_all_l2 = encode_labels(full_df, L2_COL, l2_lab2id)
y_all_l1 = encode_labels(full_df, L1_COL, l1_lab2id)

# Compatibility aliases for older cells.
y_train_l2 = y_all_l2
y_train_l1 = y_all_l1

# L2 -> L1 must be deterministic.
tmp = full_df[[L1_COL, L2_COL]].drop_duplicates()
dups = tmp.groupby(L2_COL)[L1_COL].nunique()
bad = dups[dups > 1]
if len(bad) > 0:
    raise ValueError(f"Some L2 labels map to multiple L1 parents: {bad}")

child_to_parent = tmp.set_index(L2_COL)[L1_COL].to_dict()
parent_to_children = defaultdict(list)
for child, parent in child_to_parent.items():
    parent_to_children[parent].append(child)

parent_id_for_l2 = np.zeros(num_l2, dtype=np.int64)
for l2, idx in l2_lab2id.items():
    parent_id_for_l2[idx] = l1_lab2id[child_to_parent[l2]]

children_ids_for_parent = defaultdict(list)
for l2_id, l1_id in enumerate(parent_id_for_l2):
    children_ids_for_parent[int(l1_id)].append(int(l2_id))

print("num_l2:", num_l2)
print("num_l1:", num_l1)
print("First L2 labels:", list(l2_lab2id.items())[:10])
print("Parent -> children counts:")
for p, ch in sorted(parent_to_children.items()):
    print(f"  {p}: {len(ch)}")

# Reuse the exact saved fold assignments; do not call a CV splitter here.
full_df["cv_fold"] = full_df["test_fold"].astype(int)
cv_strategy = "saved_test_fold"

if len(full_df) != full_df["cv_index"].nunique():
    raise ValueError("Saved fold data must contain one unique cv_index per requirement.")
all_test_indices = np.concatenate([
    np.flatnonzero(full_df["cv_fold"].to_numpy() == fold)
    for fold in range(N_FOLDS)
])
if len(all_test_indices) != len(full_df) or len(np.unique(all_test_indices)) != len(full_df):
    raise ValueError("Saved test folds must cover every requirement exactly once.")

print("CV strategy:", cv_strategy)
print("Validated: saved folds cover every requirement exactly once as test data.")
print(full_df["cv_fold"].value_counts().sort_index())
display(pd.crosstab(full_df["cv_fold"], full_df[L2_COL]).T.head())


In [ ]:

# ============================================================
# 4) Dataset and dataloading objects
# ============================================================

class GlobalHierarchyDataset(Dataset):
    def __init__(self, df, y_l2, y_l1, tokenizer, max_len, text_col=TEXT_COL):
        self.df = df.reset_index(drop=True)
        self.y_l2 = np.array(y_l2, dtype=np.int64)
        self.y_l1 = np.array(y_l1, dtype=np.int64)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.loc[idx, self.text_col])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.y_l2[idx], dtype=torch.long)
        item["l1_labels"] = torch.tensor(self.y_l1[idx], dtype=torch.long)
        return item


In [ ]:

# ============================================================
# 5) Weighted focal loss and class-weight helpers
# ============================================================

def compute_balanced_class_weights_flat_exact(
    label_ids,
    num_labels,
    normalize_mean=True,
    max_weight=10.0,
):
    label_ids = np.asarray(label_ids, dtype=int)
    counts = np.bincount(label_ids, minlength=num_labels).astype(np.float64)
    counts[counts == 0] = 1.0

    N = counts.sum()
    weights = N / (num_labels * counts)

    if normalize_mean:
        weights = weights / max(weights.mean(), 1e-12)

    if max_weight is not None:
        weights = np.minimum(weights, float(max_weight))

    return torch.tensor(weights, dtype=torch.float32)


class MulticlassFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()

        ce = -log_pt
        focal_factor = (1.0 - pt).pow(self.gamma)
        loss = focal_factor * ce

        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)
            loss = alpha[targets] * loss

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


# Class weights are computed inside each CV fold from the training folds only.


In [ ]:

# ============================================================
# 6) HGCLR hard-sibling-negative contrastive loss
# ============================================================
class HierarchyGuidedSupConLoss(nn.Module):
    """
    ISO hierarchy-aware supervised contrastive loss.

    Positives:
      - same L2 only

    Negatives:
      - different L1: normal negative
      - same L1 but different L2: hard sibling negative
    """
    def __init__(
        self,
        temperature: float = 0.10,
        same_l2_weight: float = 1.0,
        sibling_negative_weight: float = 1.25,
        eps: float = 1e-12,
    ):
        super().__init__()
        self.temperature = float(temperature)
        self.same_l2_weight = float(same_l2_weight)
        self.sibling_negative_weight = float(sibling_negative_weight)
        self.eps = eps

    def forward(self, features, l2_labels, l1_labels):
        device = features.device
        batch_size = features.size(0)

        if batch_size <= 1:
            return features.new_tensor(0.0)

        z = F.normalize(features, p=2, dim=-1)
        sim = torch.matmul(z, z.T) / self.temperature

        # numerical stability
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()

        eye = torch.eye(batch_size, dtype=torch.bool, device=device)
        not_self = ~eye

        l2_equal = l2_labels.unsqueeze(0) == l2_labels.unsqueeze(1)
        l1_equal = l1_labels.unsqueeze(0) == l1_labels.unsqueeze(1)

        # Positive = same exact L2 only
        same_l2_pos = l2_equal & not_self

        # Hard negative = same parent but different child
        sibling_hard_neg = l1_equal & (~l2_equal) & not_self

        # Numerator: only same-L2 positives
        pos_weights = same_l2_pos.float() * self.same_l2_weight

        # Denominator: all non-self samples, but sibling negatives are heavier
        denom_weights = torch.ones_like(sim)
        denom_weights = denom_weights + sibling_hard_neg.float() * (self.sibling_negative_weight - 1.0)
        denom_weights = denom_weights * not_self.float()

        exp_sim = torch.exp(sim) * denom_weights
        log_denom = torch.log(exp_sim.sum(dim=1, keepdim=True) + self.eps)
        log_prob = sim - log_denom

        pos_weight_sum = pos_weights.sum(dim=1)
        valid_anchor = pos_weight_sum > 0

        if valid_anchor.sum() == 0:
            return features.new_tensor(0.0)

        loss_per_anchor = -(
            (pos_weights * log_prob).sum(dim=1)
            / (pos_weight_sum + self.eps)
        )

        return loss_per_anchor[valid_anchor].mean()


In [ ]:
# ============================================================
# 7) Model: RoBERTa + L2 head + L1 head + contrastive projection + dropout
# ============================================================

class HGCLRGlobalRoberta(nn.Module):
    def __init__(
        self,
        model_name,
        num_l1,
        num_l2,
        l2_id2lab,
        l2_lab2id,
        projection_dim=256,
        enable_l1_head=True,
        enable_projection=True,
        encoder_dropout=0.15,
        attention_dropout=0.10,
        classifier_dropout=0.25,
        projection_dropout=0.20,
    ):
        super().__init__()

        self.num_l1 = num_l1
        self.num_l2 = num_l2
        self.enable_l1_head = enable_l1_head
        self.enable_projection = enable_projection

        config = AutoConfig.from_pretrained(
            model_name,
            num_labels=num_l2,
            id2label=l2_id2lab,
            label2id=l2_lab2id,
        )

        # Control dropout inside RoBERTa.
        config.hidden_dropout_prob = encoder_dropout
        config.attention_probs_dropout_prob = attention_dropout
        config.classifier_dropout = classifier_dropout

        self.seq_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            config=config,
        )

        if not hasattr(self.seq_model, "roberta"):
            raise ValueError(
                "This model class is written for roberta-base. "
                "Use roberta-base or adapt backbone access."
            )

        self.config = self.seq_model.config
        self.config.keys_to_ignore_at_inference = ["features"]

        hidden_size = self.seq_model.config.hidden_size

        if self.enable_l1_head:
            self.l1_head = nn.Sequential(
                nn.Dropout(classifier_dropout),
                nn.Linear(hidden_size, hidden_size),
                nn.Tanh(),
                nn.Dropout(classifier_dropout),
                nn.Linear(hidden_size, num_l1),
            )
        else:
            self.l1_head = None

        if self.enable_projection:
            self.contrastive_projection = nn.Sequential(
                nn.Dropout(projection_dropout),
                nn.Linear(hidden_size, hidden_size),
                nn.Tanh(),
                nn.Dropout(projection_dropout),
                nn.Linear(hidden_size, projection_dim),
            )
        else:
            self.contrastive_projection = None

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        l1_labels=None,
        **kwargs,
    ):
        outputs = self.seq_model.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        sequence_output = outputs.last_hidden_state
        cls_h = sequence_output[:, 0, :]

        # Native RoBERTa L2 classifier uses config.classifier_dropout.
        l2_logits = self.seq_model.classifier(sequence_output)

        out = {"logits": l2_logits}

        if self.enable_l1_head:
            out["l1_logits"] = self.l1_head(cls_h)

        if self.enable_projection:
            out["features"] = self.contrastive_projection(cls_h)

        return out


In [ ]:

# ============================================================
# 8) Trainer: weighted focal + L1 auxiliary + HGCLR contrastive
# ============================================================

class HGCLRGlobalTrainer(Trainer):
    def __init__(
        self,
        class_weights=None,
        gamma=2.0,
        lambda_l1=0.05,
        lambda_hgclr=0.05,
        contrast_temp=0.10,
        same_l2_weight=1.0,
        sibling_negative_weight=1.25,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = float(gamma)
        self.lambda_l1 = float(lambda_l1)
        self.lambda_hgclr = float(lambda_hgclr)

        self.focal_loss = MulticlassFocalLoss(
            alpha=class_weights,
            gamma=self.gamma,
            reduction="mean",
        )


        self.hgclr_loss = HierarchyGuidedSupConLoss(
            temperature=contrast_temp,
            same_l2_weight=same_l2_weight,
            sibling_negative_weight=sibling_negative_weight,
            )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        l1_labels = inputs.get("l1_labels")

        model_inputs = {
            k: v for k, v in inputs.items()
            if k not in ["labels", "l1_labels"]
        }

        outputs = model(**model_inputs)
        logits = outputs["logits"]

        # Main L2 loss.
        l2_loss = self.focal_loss(logits, labels)
        loss = l2_loss

        # L1 auxiliary loss.
        if self.lambda_l1 > 0:
            if "l1_logits" not in outputs:
                raise ValueError("lambda_l1 > 0 but model did not return l1_logits.")
            if l1_labels is None:
                raise ValueError("lambda_l1 > 0 requires l1_labels.")
            l1_loss = F.cross_entropy(outputs["l1_logits"], l1_labels)
            loss = loss + self.lambda_l1 * l1_loss

        # HGCLR-style hierarchy-guided contrastive loss.
        if self.lambda_hgclr > 0:
            if "features" not in outputs:
                raise ValueError("lambda_hgclr > 0 but model did not return features.")
            hgclr_loss = self.hgclr_loss(outputs["features"], labels, l1_labels)
            loss = loss + self.lambda_hgclr * hgclr_loss

        return (loss, outputs) if return_outputs else loss


In [ ]:

# ============================================================
# 9) Prediction extraction helpers
# ============================================================

def extract_predictions_tuple(predictions):
    """
    Expected order from model output dict:
        0: logits      [L2]
        1: l1_logits   [L1]
        2: features    [may be ignored by HF]
    """
    if isinstance(predictions, (tuple, list)):
        l2_logits = np.asarray(predictions[0])
        l1_logits = np.asarray(predictions[1]) if len(predictions) > 1 else None
    elif isinstance(predictions, dict):
        l2_logits = np.asarray(predictions["logits"])
        l1_logits = np.asarray(predictions.get("l1_logits")) if "l1_logits" in predictions else None
    else:
        l2_logits = np.asarray(predictions)
        l1_logits = None
    return l2_logits, l1_logits


def extract_label_tuple(label_ids):
    if isinstance(label_ids, (tuple, list)):
        y_l2 = np.asarray(label_ids[0])
        y_l1 = np.asarray(label_ids[1]) if len(label_ids) > 1 else None
    else:
        y_l2 = np.asarray(label_ids)
        y_l1 = None
    return y_l2, y_l1


def make_compute_metrics_hgclr():
    def compute_metrics(eval_pred):
        l2_logits, l1_logits = extract_predictions_tuple(eval_pred.predictions)
        y_l2, y_l1 = extract_label_tuple(eval_pred.label_ids)

        pred_l2 = np.argmax(l2_logits, axis=1)
        out = {
            "macro_f1": float(f1_score(y_l2, pred_l2, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_l2, pred_l2, average="weighted", zero_division=0)),
            "accuracy": float(accuracy_score(y_l2, pred_l2)),
            "balanced_acc": float(balanced_accuracy_score(y_l2, pred_l2)),
            "precision_macro": float(precision_score(y_l2, pred_l2, average="macro", zero_division=0)),
            "recall_macro": float(recall_score(y_l2, pred_l2, average="macro", zero_division=0)),
        }

        # Validation-time parent/path diagnostics.
        if y_l1 is not None:
            pred_l1_from_l2 = parent_id_for_l2[pred_l2]
            out["parent_from_l2_acc"] = float(accuracy_score(y_l1, pred_l1_from_l2))
            h = compute_path_metrics(y_l2, pred_l2, y_l1, pred_l1_from_l2)
            out["HP"] = h["h_precision"]
            out["HR"] = h["h_recall"]
            out["HF1"] = h["h_f1"]
            out["wrong_l1_pct"] = h["wrong_l1_pct"]
        else:
            pred_l1_from_l2 = None

        if l1_logits is not None and y_l1 is not None:
            pred_l1_aux = np.argmax(l1_logits, axis=1)
            out["l1_aux_macro_f1"] = float(f1_score(y_l1, pred_l1_aux, average="macro", zero_division=0))
            out["l1_aux_accuracy"] = float(accuracy_score(y_l1, pred_l1_aux))
            out["parent_child_consistency_aux"] = float(np.mean(pred_l1_aux == pred_l1_from_l2))

        return out
    return compute_metrics


In [ ]:

# ============================================================
# 10) Path-guided routing decoders: base, hard mask, soft fusion, top-k
# ============================================================

def _np_log_softmax(x, axis=-1):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x, axis=axis, keepdims=True)
    z = x - m
    return z - np.log(np.sum(np.exp(z), axis=axis, keepdims=True) + 1e-12)


def get_decoder_name(decoder_cfg):
    mode = decoder_cfg.get("decoder", "base_l2")
    top_k = decoder_cfg.get("top_k")
    alpha = decoder_cfg.get("alpha")
    if mode == "topk_parent":
        return f"topk_parent_{int(top_k)}"
    if mode == "soft_fusion":
        return "soft_fusion"
    return mode


def attach_decoder_metadata(df, decoder_cfg, decoder_name):
    df = df.copy()
    df["decoder"] = decoder_cfg.get("decoder", "base_l2")
    df["decoder_name"] = decoder_name
    df["decoder_top_k"] = decoder_cfg.get("top_k")
    df["decoder_alpha"] = decoder_cfg.get("alpha")
    return df


def decode_l2_predictions_with_routing(l2_logits, l1_logits=None, decoder="base_l2", top_k=None, alpha=1.0):
    """
    Convert L2 and L1 logits into final L2 predictions.

    Decoders:
      base_l2: direct argmax over L2 logits.
      hard_mask: predict L1 by the auxiliary head, then allow only children of that parent.
      soft_fusion: add parent log-probability to each L2 logit: score(l2)=l2_logit + alpha*log P(parent(l2)).
      topk_parent: allow children whose parent is among the top-k predicted L1 parents.

    These are inference-only decoders. They do not change HGCLR training.
    """
    l2_logits = np.asarray(l2_logits)
    if decoder == "base_l2" or l1_logits is None:
        return np.argmax(l2_logits, axis=1)

    l1_logits = np.asarray(l1_logits)
    parent_for_l2 = np.asarray(parent_id_for_l2, dtype=int)

    if decoder == "hard_mask":
        pred_parent = np.argmax(l1_logits, axis=1)
        scores = l2_logits.astype(np.float64).copy()
        allowed = parent_for_l2[None, :] == pred_parent[:, None]
        scores[~allowed] = -1e9
        return np.argmax(scores, axis=1)

    if decoder == "soft_fusion":
        if alpha is None:
            alpha = 1.0
        l1_log_probs = _np_log_softmax(l1_logits, axis=1)
        parent_log_probs_for_l2 = l1_log_probs[:, parent_for_l2]
        scores = l2_logits.astype(np.float64) + float(alpha) * parent_log_probs_for_l2
        return np.argmax(scores, axis=1)

    if decoder == "topk_parent":
        if top_k is None:
            top_k = 2
        top_k = int(top_k)
        if top_k >= l1_logits.shape[1]:
            return np.argmax(l2_logits, axis=1)
        top_parents = np.argsort(-l1_logits, axis=1)[:, :top_k]
        scores = l2_logits.astype(np.float64).copy()
        allowed = np.zeros_like(scores, dtype=bool)
        for i in range(scores.shape[0]):
            allowed[i] = np.isin(parent_for_l2, top_parents[i])
        scores[~allowed] = -1e9
        return np.argmax(scores, axis=1)

    raise ValueError(f"Unknown decoder: {decoder}")


In [ ]:

# ============================================================
# 11) TrainingArguments builder and checkpoint callback
# ============================================================

class FixLayerNormCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        model_file = os.path.join(ckpt_dir, "pytorch_model.bin")
        if not os.path.exists(model_file):
            return control
        try:
            sd = torch.load(model_file, map_location="cpu")
            changed = False
            new_sd = dict(sd)
            for k, v in sd.items():
                if k.endswith("LayerNorm.gamma"):
                    nk = k.replace("LayerNorm.gamma", "LayerNorm.weight")
                    if nk not in new_sd:
                        new_sd[nk] = v
                        changed = True
                elif k.endswith("LayerNorm.beta"):
                    nk = k.replace("LayerNorm.beta", "LayerNorm.bias")
                    if nk not in new_sd:
                        new_sd[nk] = v
                        changed = True
            if changed:
                torch.save(new_sd, model_file)
                print(f"Fixed LayerNorm aliases in {ckpt_dir}")
        except Exception as e:
            print(f"Could not fix checkpoint {ckpt_dir}: {e}")
        return control


def make_hgclr_training_args(run_dir, seed, train_ds, batch_size=BATCH_SIZE, lr=LR, wd=WD):
    steps_per_epoch = int(math.ceil(len(train_ds) / batch_size))
    total_steps = steps_per_epoch * MAX_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)

    print("steps_per_epoch:", steps_per_epoch)
    print("total_steps:", total_steps)
    print("warmup_steps:", warmup_steps)

    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters

    kwargs = dict(
        output_dir=run_dir,
        seed=seed,
        data_seed=seed,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=MAX_EPOCHS,
        save_strategy="epoch",
        optim="adamw_torch",
        learning_rate=lr,
        weight_decay=wd,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        max_grad_norm=MAX_GRAD_NORM,
        logging_strategy="epoch",
        load_best_model_at_end=USE_ROTATING_VALIDATION_FOLD,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        fp16=torch.cuda.is_available(),
        full_determinism=True,
        remove_unused_columns=False,
        label_names=["labels", "l1_labels"],
    )

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "epoch" if USE_ROTATING_VALIDATION_FOLD else "no"
    else:
        kwargs["evaluation_strategy"] = "epoch" if USE_ROTATING_VALIDATION_FOLD else "no"

    kwargs = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**kwargs)


In [ ]:

# ============================================================
# 12) Metrics: flat metrics, HP, HR, HF1, and error decomposition
# ============================================================

def compute_path_metrics(true_l2, pred_l2, true_l1, pred_l1_from_l2):
    """
    Hierarchical path metric using consistent paths derived from L2:
        true path = {true L1, true L2}
        pred path = {parent(pred L2), pred L2}

    Score per sample:
        correct L2 -> 1.0
        wrong L2 but correct L1 -> 0.5
        wrong L1 -> 0.0
    """
    true_l2 = np.asarray(true_l2)
    pred_l2 = np.asarray(pred_l2)
    true_l1 = np.asarray(true_l1)
    pred_l1_from_l2 = np.asarray(pred_l1_from_l2)

    correct_l2 = pred_l2 == true_l2
    same_parent = pred_l1_from_l2 == true_l1

    intersection = np.zeros(len(true_l2), dtype=float)
    intersection[correct_l2] = 2.0
    intersection[(~correct_l2) & same_parent] = 1.0

    h_precision = intersection / 2.0
    h_recall = intersection / 2.0
    h_f1 = np.where(intersection > 0, 2 * h_precision * h_recall / (h_precision + h_recall + 1e-12), 0.0)

    hier_distance = np.zeros(len(true_l2), dtype=float)
    hier_distance[(~correct_l2) & same_parent] = 1.0
    hier_distance[~same_parent] = 2.0

    return {
        "h_precision": float(np.mean(h_precision)),
        "h_recall": float(np.mean(h_recall)),
        "h_f1": float(np.mean(h_f1)),
        "mean_hier_distance": float(np.mean(hier_distance)),
        "mean_hier_distance_norm": float(np.mean(hier_distance) / 2.0),
        "correct_l2_count": int(np.sum(correct_l2)),
        "wrong_l2_correct_l1_count": int(np.sum((~correct_l2) & same_parent)),
        "wrong_l1_count": int(np.sum(~same_parent)),
        "correct_l2_pct": float(np.mean(correct_l2)),
        "wrong_l2_correct_l1_pct": float(np.mean((~correct_l2) & same_parent)),
        "wrong_l1_pct": float(np.mean(~same_parent)),
    }



def compute_overall_metrics(y_l2, pred_l2, y_l1, pred_l1_aux=None, prefix="test_"):
    pred_l1_from_l2 = parent_id_for_l2[pred_l2]

    out = {
        f"{prefix}macro_f1": float(f1_score(y_l2, pred_l2, average="macro", zero_division=0)),
        f"{prefix}weighted_f1": float(f1_score(y_l2, pred_l2, average="weighted", zero_division=0)),
        f"{prefix}accuracy": float(accuracy_score(y_l2, pred_l2)),
        f"{prefix}balanced_acc": float(balanced_accuracy_score(y_l2, pred_l2)),
        f"{prefix}precision_macro": float(precision_score(y_l2, pred_l2, average="macro", zero_division=0)),
        f"{prefix}recall_macro": float(recall_score(y_l2, pred_l2, average="macro", zero_division=0)),
        f"{prefix}parent_from_l2_macro_f1": float(f1_score(y_l1, pred_l1_from_l2, average="macro", zero_division=0)),
        f"{prefix}parent_from_l2_weighted_f1": float(f1_score(y_l1, pred_l1_from_l2, average="weighted", zero_division=0)),
        f"{prefix}parent_from_l2_acc": float(accuracy_score(y_l1, pred_l1_from_l2)),
        f"{prefix}parent_from_l2_balanced_acc": float(balanced_accuracy_score(y_l1, pred_l1_from_l2)),
        f"{prefix}parent_from_l2_precision_macro": float(precision_score(y_l1, pred_l1_from_l2, average="macro", zero_division=0)),
        f"{prefix}parent_from_l2_recall_macro": float(recall_score(y_l1, pred_l1_from_l2, average="macro", zero_division=0)),
    }

    h = compute_path_metrics(y_l2, pred_l2, y_l1, pred_l1_from_l2)
    out.update({f"{prefix}" + k: v for k, v in h.items()})

    # Reviewer-friendly aliases for hierarchical metrics.
    out[f"{prefix}HP"] = h["h_precision"]
    out[f"{prefix}HR"] = h["h_recall"]
    out[f"{prefix}HF1"] = h["h_f1"]

    if pred_l1_aux is not None:
        out.update({
            f"{prefix}l1_aux_macro_f1": float(f1_score(y_l1, pred_l1_aux, average="macro", zero_division=0)),
            f"{prefix}l1_aux_weighted_f1": float(f1_score(y_l1, pred_l1_aux, average="weighted", zero_division=0)),
            f"{prefix}l1_aux_accuracy": float(accuracy_score(y_l1, pred_l1_aux)),
            f"{prefix}l1_aux_balanced_acc": float(balanced_accuracy_score(y_l1, pred_l1_aux)),
            f"{prefix}parent_child_consistency_aux": float(np.mean(pred_l1_aux == pred_l1_from_l2)),
        })

    return out

def make_prediction_dataframe(base_df, y_l2, pred_l2, y_l1, pred_l1_aux=None, split="test"):
    pred_l1_from_l2 = parent_id_for_l2[pred_l2]
    df = base_df.copy().reset_index(drop=True)

    df["split"] = split
    df["true_l1_id"] = y_l1
    df["true_l2_id"] = y_l2
    df["pred_l2_id"] = pred_l2
    df["pred_l1_from_l2_id"] = pred_l1_from_l2

    df["true_l1"] = [l1_id2lab[int(i)] for i in y_l1]
    df["true_l2"] = [l2_id2lab[int(i)] for i in y_l2]
    df["pred_l2"] = [l2_id2lab[int(i)] for i in pred_l2]
    df["pred_l1_from_l2"] = [l1_id2lab[int(i)] for i in pred_l1_from_l2]

    if pred_l1_aux is not None:
        df["pred_l1_aux_id"] = pred_l1_aux
        df["pred_l1_aux"] = [l1_id2lab[int(i)] for i in pred_l1_aux]
        df["aux_l1_correct"] = df["pred_l1_aux_id"] == df["true_l1_id"]
        df["aux_l1_l2_parent_consistent"] = df["pred_l1_aux_id"] == df["pred_l1_from_l2_id"]

    df["l2_correct"] = df["pred_l2_id"] == df["true_l2_id"]
    df["parent_from_l2_correct"] = df["pred_l1_from_l2_id"] == df["true_l1_id"]
    df["wrong_l2_but_correct_l1"] = (~df["l2_correct"]) & df["parent_from_l2_correct"]
    df["wrong_l1"] = ~df["parent_from_l2_correct"]

    return df


In [ ]:

# ============================================================
# 13) Plotting, confusion matrices, per-class and per-parent helpers
# ============================================================

def per_class_analysis(y_true, y_pred, seed, experiment_name, split="test"):
    rep = classification_report(
        y_true,
        y_pred,
        labels=list(range(num_l2)),
        target_names=[l2_id2lab[i] for i in range(num_l2)],
        output_dict=True,
        zero_division=0,
    )

    rows = []
    for l2_id in range(num_l2):
        lab = l2_id2lab[l2_id]
        d = rep[lab]
        rows.append({
            "experiment_name": experiment_name,
            "seed": seed,
            "split": split,
            "l2_id": l2_id,
            "L2": lab,
            "L1": child_to_parent[lab],
            "support": int(d["support"]),
            "precision": float(d["precision"]),
            "recall": float(d["recall"]),
            "f1": float(d["f1-score"]),
        })
    return pd.DataFrame(rows)


def per_parent_analysis(y_l2, pred_l2, y_l1, pred_l1_aux, seed, experiment_name, split="test"):
    rows = []
    pred_l1_from_l2 = parent_id_for_l2[pred_l2]

    for l1_id in range(num_l1):
        mask = y_l1 == l1_id
        if mask.sum() == 0:
            continue

        child_ids = children_ids_for_parent[int(l1_id)]
        parent_name = l1_id2lab[int(l1_id)]

        row = {
            "experiment_name": experiment_name,
            "seed": seed,
            "split": split,
            "L1": parent_name,
            "l1_id": int(l1_id),
            "support": int(mask.sum()),
            "n_child_l2_train": int(len(child_ids)),
            "l2_accuracy_within_true_parent": float(accuracy_score(y_l2[mask], pred_l2[mask])),
            "l2_macro_f1_within_true_parent": float(f1_score(
                y_l2[mask], pred_l2[mask], labels=child_ids, average="macro", zero_division=0
            )),
            "parent_from_l2_accuracy": float(accuracy_score(y_l1[mask], pred_l1_from_l2[mask])),
            "wrong_parent_rate": float(np.mean(pred_l1_from_l2[mask] != y_l1[mask])),
            "wrong_l2_but_correct_parent_rate": float(np.mean((pred_l2[mask] != y_l2[mask]) & (pred_l1_from_l2[mask] == y_l1[mask]))),
        }

        if pred_l1_aux is not None:
            row["l1_aux_accuracy"] = float(accuracy_score(y_l1[mask], pred_l1_aux[mask]))
            row["aux_parent_child_consistency"] = float(np.mean(pred_l1_aux[mask] == pred_l1_from_l2[mask]))

        rows.append(row)

    return pd.DataFrame(rows)


def save_confusion_matrices(y_l2, pred_l2, y_l1, pred_l1_aux, run_dir, prefix="test"):
    # L2 confusion matrix
    cm_l2 = confusion_matrix(y_l2, pred_l2, labels=list(range(num_l2)))
    cm_l2_df = pd.DataFrame(
        cm_l2,
        index=[f"true_{l2_id2lab[i]}" for i in range(num_l2)],
        columns=[f"pred_{l2_id2lab[i]}" for i in range(num_l2)],
    )
    cm_l2_path = os.path.join(run_dir, f"{prefix}_confusion_matrix_L2.csv")
    cm_l2_df.to_csv(cm_l2_path)

    # Parent confusion matrix from predicted L2 parent.
    pred_l1_from_l2 = parent_id_for_l2[pred_l2]
    cm_l1 = confusion_matrix(y_l1, pred_l1_from_l2, labels=list(range(num_l1)))
    cm_l1_df = pd.DataFrame(
        cm_l1,
        index=[f"true_{l1_id2lab[i]}" for i in range(num_l1)],
        columns=[f"pred_{l1_id2lab[i]}" for i in range(num_l1)],
    )
    cm_l1_path = os.path.join(run_dir, f"{prefix}_confusion_matrix_L1_from_L2.csv")
    cm_l1_df.to_csv(cm_l1_path)

    paths = {
        "cm_l2_path": cm_l2_path,
        "cm_l1_from_l2_path": cm_l1_path,
    }

    if pred_l1_aux is not None:
        cm_l1_aux = confusion_matrix(y_l1, pred_l1_aux, labels=list(range(num_l1)))
        cm_l1_aux_df = pd.DataFrame(
            cm_l1_aux,
            index=[f"true_{l1_id2lab[i]}" for i in range(num_l1)],
            columns=[f"pred_{l1_id2lab[i]}" for i in range(num_l1)],
        )
        cm_l1_aux_path = os.path.join(run_dir, f"{prefix}_confusion_matrix_L1_aux.csv")
        cm_l1_aux_df.to_csv(cm_l1_aux_path)
        paths["cm_l1_aux_path"] = cm_l1_aux_path

    return paths


# ============================================================
# Plot helpers for CV reporting
# ============================================================

def save_cm_plot(cm, row_labels, col_labels, png_path, title, figsize=(14, 12)):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, interpolation="nearest", aspect="auto")
    ax.figure.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_xticklabels(col_labels, rotation=90, fontsize=7)
    ax.set_yticklabels(row_labels, fontsize=7)

    if cm.shape[0] <= 12 and cm.shape[1] <= 12:
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center")

    fig.tight_layout()
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return png_path


def save_confusion_matrices_with_plots(y_l2, pred_l2, y_l1, pred_l1_aux, run_dir, prefix="test"):
    paths = save_confusion_matrices(y_l2, pred_l2, y_l1, pred_l1_aux, run_dir, prefix=prefix)

    cm_l2 = confusion_matrix(y_l2, pred_l2, labels=list(range(num_l2)))
    paths["cm_l2_png"] = save_cm_plot(
        cm_l2,
        [f"true_{l2_id2lab[i]}" for i in range(num_l2)],
        [f"pred_{l2_id2lab[i]}" for i in range(num_l2)],
        os.path.join(run_dir, f"{prefix}_confusion_matrix_L2.png"),
        f"{prefix} L2 confusion matrix",
        figsize=(18, 16),
    )

    pred_l1_from_l2 = parent_id_for_l2[pred_l2]
    cm_l1 = confusion_matrix(y_l1, pred_l1_from_l2, labels=list(range(num_l1)))
    paths["cm_l1_from_l2_png"] = save_cm_plot(
        cm_l1,
        [f"true_{l1_id2lab[i]}" for i in range(num_l1)],
        [f"pred_{l1_id2lab[i]}" for i in range(num_l1)],
        os.path.join(run_dir, f"{prefix}_confusion_matrix_L1_from_L2.png"),
        f"{prefix} parent confusion matrix from predicted L2",
        figsize=(8, 7),
    )

    if pred_l1_aux is not None:
        cm_l1_aux = confusion_matrix(y_l1, pred_l1_aux, labels=list(range(num_l1)))
        paths["cm_l1_aux_png"] = save_cm_plot(
            cm_l1_aux,
            [f"true_{l1_id2lab[i]}" for i in range(num_l1)],
            [f"pred_{l1_id2lab[i]}" for i in range(num_l1)],
            os.path.join(run_dir, f"{prefix}_confusion_matrix_L1_aux.png"),
            f"{prefix} auxiliary L1 confusion matrix",
            figsize=(8, 7),
        )

    return paths


def save_loss_plots_from_trainer(trainer, run_dir, prefix="fold"):
    log_df = pd.DataFrame(trainer.state.log_history)
    log_path = os.path.join(run_dir, f"{prefix}_trainer_log_history.csv")
    log_df.to_csv(log_path, index=False)

    plot_path = os.path.join(run_dir, f"{prefix}_train_eval_loss.png")
    fig, ax = plt.subplots(figsize=(9, 5))

    if "epoch" in log_df.columns and "loss" in log_df.columns:
        tr = log_df.dropna(subset=["loss"])
        if not tr.empty:
            ax.plot(tr["epoch"], tr["loss"], marker="o", label="train_loss")

    if "epoch" in log_df.columns and "eval_loss" in log_df.columns:
        ev = log_df.dropna(subset=["eval_loss"])
        if not ev.empty:
            ax.plot(ev["epoch"], ev["eval_loss"], marker="o", label="eval_loss")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training and validation loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return log_path, plot_path


In [ ]:
def diagnose_contrastive_batch_coverage(y_l2, y_l1, batch_size=16, n_trials=200, seed=42):
    rng = np.random.default_rng(seed)
    y_l2 = np.asarray(y_l2)
    y_l1 = np.asarray(y_l1)
    n = len(y_l2)

    stats = []

    for _ in range(n_trials):
        perm = rng.permutation(n)

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            if len(idx) <= 1:
                continue

            b_l2 = y_l2[idx]
            b_l1 = y_l1[idx]
            B = len(idx)

            l2_equal = b_l2[:, None] == b_l2[None, :]
            l1_equal = b_l1[:, None] == b_l1[None, :]
            not_self = ~np.eye(B, dtype=bool)

            pos = l2_equal & not_self
            sibling_neg = l1_equal & (~l2_equal) & not_self
            cross_parent_neg = (~l1_equal) & not_self

            pos_count = pos.sum(axis=1)
            sibling_count = sibling_neg.sum(axis=1)
            cross_count = cross_parent_neg.sum(axis=1)

            stats.append({
                "batch_size": B,
                "valid_anchor_rate": float((pos_count > 0).mean()),
                "avg_pos_per_anchor": float(pos_count.mean()),
                "avg_sibling_neg_per_anchor": float(sibling_count.mean()),
                "anchor_with_sibling_neg_rate": float((sibling_count > 0).mean()),
                "avg_cross_parent_neg_per_anchor": float(cross_count.mean()),
                "batch_has_any_positive": bool(pos.sum() > 0),
                "batch_has_any_sibling_negative": bool(sibling_neg.sum() > 0),
                "n_unique_l2": int(len(np.unique(b_l2))),
                "n_unique_l1": int(len(np.unique(b_l1))),
            })

    df = pd.DataFrame(stats)

    summary = {
        "valid_anchor_rate_mean": df["valid_anchor_rate"].mean(),
        "avg_pos_per_anchor_mean": df["avg_pos_per_anchor"].mean(),
        "avg_sibling_neg_per_anchor_mean": df["avg_sibling_neg_per_anchor"].mean(),
        "anchor_with_sibling_neg_rate_mean": df["anchor_with_sibling_neg_rate"].mean(),
        "avg_cross_parent_neg_per_anchor_mean": df["avg_cross_parent_neg_per_anchor"].mean(),
        "batch_without_any_positive_pct": 1.0 - df["batch_has_any_positive"].mean(),
        "batch_without_any_sibling_negative_pct": 1.0 - df["batch_has_any_sibling_negative"].mean(),
        "avg_unique_l2_per_batch": df["n_unique_l2"].mean(),
        "avg_unique_l1_per_batch": df["n_unique_l1"].mean(),
        "encoder_dropout": ENCODER_DROPOUT,
        "attention_dropout": ATTENTION_DROPOUT,
        "classifier_dropout": CLASSIFIER_DROPOUT,
        "projection_dropout": PROJECTION_DROPOUT,
    }

    return pd.Series(summary), df

In [ ]:

# ============================================================
# 14) Run one CV fold for one HGCLR config
# ============================================================

def get_fold_splits(test_fold):
    val_fold = (test_fold + 1) % N_FOLDS if USE_ROTATING_VALIDATION_FOLD else None

    test_mask = full_df["cv_fold"].values == test_fold
    if USE_ROTATING_VALIDATION_FOLD:
        val_mask = full_df["cv_fold"].values == val_fold
        train_mask = ~(test_mask | val_mask)
    else:
        val_mask = np.zeros(len(full_df), dtype=bool)
        train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    val_idx = np.where(val_mask)[0]
    test_idx = np.where(test_mask)[0]
    return train_idx, val_idx, test_idx, val_fold


def run_one_cv_fold_hgclr(cfg, test_fold):
    print("\n" + "=" * 100)
    print(f"RUN: {cfg['experiment_name']} | seed={SEED} | test_fold={test_fold}")
    print("=" * 100)

    set_seed(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_idx, val_idx, test_idx, val_fold = get_fold_splits(test_fold)

    train_fold_df = full_df.iloc[train_idx].reset_index(drop=True)
    val_fold_df   = full_df.iloc[val_idx].reset_index(drop=True) if len(val_idx) else None
    test_fold_df  = full_df.iloc[test_idx].reset_index(drop=True)

    y_train_l2_fold = y_all_l2[train_idx]
    y_train_l1_fold = y_all_l1[train_idx]
    y_val_l2_fold = y_all_l2[val_idx] if len(val_idx) else None
    y_val_l1_fold = y_all_l1[val_idx] if len(val_idx) else None
    y_test_l2_fold = y_all_l2[test_idx]
    y_test_l1_fold = y_all_l1[test_idx]

    print("Train/Val/Test sizes:", len(train_idx), len(val_idx), len(test_idx))
    print("Val fold:", val_fold)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    train_ds = GlobalHierarchyDataset(train_fold_df, y_train_l2_fold, y_train_l1_fold, tokenizer, MAX_LEN, TEXT_COL)
    val_ds   = GlobalHierarchyDataset(val_fold_df, y_val_l2_fold, y_val_l1_fold, tokenizer, MAX_LEN, TEXT_COL) if len(val_idx) else None
    test_ds  = GlobalHierarchyDataset(test_fold_df, y_test_l2_fold, y_test_l1_fold, tokenizer, MAX_LEN, TEXT_COL)

    class_w = compute_balanced_class_weights_flat_exact(
        y_train_l2_fold,
        num_labels=num_l2,
        normalize_mean=True,
        max_weight=MAX_CLASS_WEIGHT_CLIP,
    )

    run_name = f"{cfg['experiment_name']}_seed{SEED}_fold{test_fold}"
    run_dir = os.path.join(OUTPUT_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)


    #Check the Contrastive Examples Coverage,
    coverage_summary, coverage_batches = diagnose_contrastive_batch_coverage(
        y_train_l2_fold,
        y_train_l1_fold,
        batch_size=BATCH_SIZE,
        n_trials=200,
        seed=SEED + test_fold,
        )
    print("\nContrastive batch coverage:")
    print(coverage_summary)
    coverage_summary.to_csv(os.path.join(run_dir, "contrastive_batch_coverage_summary.csv"))
    coverage_batches.to_csv(os.path.join(run_dir, "contrastive_batch_coverage_batches.csv"), index=False)
     # End check


    # Remove old checkpoints from same run folder to avoid mixing.
    for item in os.listdir(run_dir):
        p = os.path.join(run_dir, item)
        if item.startswith("checkpoint-") and os.path.isdir(p):
            shutil.rmtree(p)
            print(f"Removed old checkpoint: {p}")

    model = HGCLRGlobalRoberta(
        model_name=MODEL_NAME,
        num_l1=num_l1,
        num_l2=num_l2,
        l2_id2lab=l2_id2lab,
        l2_lab2id=l2_lab2id,
        projection_dim=cfg.get("projection_dim", 256),
        enable_l1_head=cfg.get("lambda_l1", 0.0) > 0,
        enable_projection=cfg.get("lambda_hgclr", 0.0) > 0,
        encoder_dropout=ENCODER_DROPOUT,
        attention_dropout=ATTENTION_DROPOUT,
        classifier_dropout=CLASSIFIER_DROPOUT,
        projection_dropout=PROJECTION_DROPOUT,
    )

    args = make_hgclr_training_args(run_dir, SEED, train_ds, BATCH_SIZE, LR, WD)

    callbacks = [FixLayerNormCheckpointCallback()]
    if USE_ROTATING_VALIDATION_FOLD:
        callbacks.insert(0, EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE))

    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=make_compute_metrics_hgclr() if USE_ROTATING_VALIDATION_FOLD else None,
        callbacks=callbacks,
    )

    trainer_init = dict(
        class_weights=class_w,
        gamma=FOCAL_GAMMA,
        lambda_l1=cfg.get("lambda_l1", 0.05),
        lambda_hgclr=cfg.get("lambda_hgclr", 0.05),
        contrast_temp=cfg.get("contrast_temp", 0.10),
        same_l2_weight=cfg.get("same_l2_weight", 1.0),
        sibling_negative_weight=cfg.get("sibling_negative_weight", 1.25),
    )

    try:
        trainer = HGCLRGlobalTrainer(
            **trainer_kwargs,
            processing_class=tokenizer,
            **trainer_init,
        )
    except TypeError:
        trainer = HGCLRGlobalTrainer(
            **trainer_kwargs,
            tokenizer=tokenizer,
            **trainer_init,
        )

    trainer.train()

    loss_log_path, loss_plot_path = save_loss_plots_from_trainer(trainer, run_dir, prefix="fold")

    # Base eval metrics used by early stopping/best checkpoint. Decoder-specific validation metrics are computed below.
    eval_metrics_from_trainer = {}
    if USE_ROTATING_VALIDATION_FOLD and val_ds is not None:
        eval_metrics_from_trainer = trainer.evaluate(val_ds)

    val_l2_logits = val_l1_logits = val_y_l2 = val_y_l1 = None
    if val_ds is not None:
        val_output = trainer.predict(val_ds)
        val_l2_logits, val_l1_logits = extract_predictions_tuple(val_output.predictions)
        val_y_l2, val_y_l1 = extract_label_tuple(val_output.label_ids)

    test_output = trainer.predict(test_ds)
    test_l2_logits, test_l1_logits = extract_predictions_tuple(test_output.predictions)
    test_y_l2, test_y_l1 = extract_label_tuple(test_output.label_ids)
    test_pred_l1_aux = np.argmax(test_l1_logits, axis=1) if test_l1_logits is not None else None
    val_pred_l1_aux = np.argmax(val_l1_logits, axis=1) if val_l1_logits is not None else None

    summary_rows = []
    pred_dfs = []
    per_class_dfs = []
    per_parent_dfs = []
    hier_error_dfs = []

    for decoder_cfg in DECODER_CONFIGS:
        decoder_name = get_decoder_name(decoder_cfg)
        decoder_dir = os.path.join(run_dir, f"decoder_{decoder_name}")
        os.makedirs(decoder_dir, exist_ok=True)

        test_pred_l2 = decode_l2_predictions_with_routing(
            test_l2_logits,
            test_l1_logits,
            decoder=decoder_cfg.get("decoder", "base_l2"),
            top_k=decoder_cfg.get("top_k"),
            alpha=decoder_cfg.get("alpha", 1.0),
        )

        val_metrics = {}
        if val_l2_logits is not None:
            val_pred_l2 = decode_l2_predictions_with_routing(
                val_l2_logits,
                val_l1_logits,
                decoder=decoder_cfg.get("decoder", "base_l2"),
                top_k=decoder_cfg.get("top_k"),
                alpha=decoder_cfg.get("alpha", 1.0),
            )
            val_metrics = compute_overall_metrics(
                y_l2=val_y_l2,
                pred_l2=val_pred_l2,
                y_l1=val_y_l1,
                pred_l1_aux=val_pred_l1_aux,
                prefix="val_",
            )

        test_metrics = compute_overall_metrics(
            y_l2=test_y_l2,
            pred_l2=test_pred_l2,
            y_l1=test_y_l1,
            pred_l1_aux=test_pred_l1_aux,
            prefix="test_",
        )

        pred_df = make_prediction_dataframe(
            base_df=test_fold_df,
            y_l2=test_y_l2,
            pred_l2=test_pred_l2,
            y_l1=test_y_l1,
            pred_l1_aux=test_pred_l1_aux,
            split="test",
        )
        pred_df["experiment_name"] = cfg["experiment_name"]
        pred_df["seed"] = SEED
        pred_df["fold"] = test_fold
        pred_df["val_fold"] = val_fold
        pred_df["is_oof"] = True
        pred_df = attach_decoder_metadata(pred_df, decoder_cfg, decoder_name)

        pred_path = os.path.join(decoder_dir, "test_predictions_oof_hgclr.csv")
        pred_df.to_csv(pred_path, index=False)

        report_df = pd.DataFrame(classification_report(
            test_y_l2,
            test_pred_l2,
            labels=list(range(num_l2)),
            target_names=[l2_id2lab[i] for i in range(num_l2)],
            zero_division=0,
            output_dict=True,
        )).T
        report_path = os.path.join(decoder_dir, "classification_report_test_L2.csv")
        report_df.to_csv(report_path)

        cm_paths = save_confusion_matrices_with_plots(
            y_l2=test_y_l2,
            pred_l2=test_pred_l2,
            y_l1=test_y_l1,
            pred_l1_aux=test_pred_l1_aux,
            run_dir=decoder_dir,
            prefix="test",
        )

        per_class_df = per_class_analysis(test_y_l2, test_pred_l2, SEED, cfg["experiment_name"], split="test")
        per_class_df["fold"] = test_fold
        per_class_df["val_fold"] = val_fold
        per_class_df = attach_decoder_metadata(per_class_df, decoder_cfg, decoder_name)
        per_class_path = os.path.join(decoder_dir, "per_class_test_L2.csv")
        per_class_df.to_csv(per_class_path, index=False)

        per_parent_df = per_parent_analysis(test_y_l2, test_pred_l2, test_y_l1, test_pred_l1_aux, SEED, cfg["experiment_name"], split="test")
        per_parent_df["fold"] = test_fold
        per_parent_df["val_fold"] = val_fold
        per_parent_df = attach_decoder_metadata(per_parent_df, decoder_cfg, decoder_name)
        per_parent_path = os.path.join(decoder_dir, "per_parent_test.csv")
        per_parent_df.to_csv(per_parent_path, index=False)

        hier_error = {
            "experiment_name": cfg["experiment_name"],
            "seed": SEED,
            "fold": test_fold,
            "val_fold": val_fold,
            "split": "test",
            "decoder": decoder_cfg.get("decoder", "base_l2"),
            "decoder_name": decoder_name,
            "decoder_top_k": decoder_cfg.get("top_k"),
            "decoder_alpha": decoder_cfg.get("alpha"),
            "n": int(len(test_y_l2)),
            "correct_l2_count": int(test_metrics["test_correct_l2_count"]),
            "wrong_l2_correct_l1_count": int(test_metrics["test_wrong_l2_correct_l1_count"]),
            "wrong_l1_count": int(test_metrics["test_wrong_l1_count"]),
            "correct_l2_pct": float(test_metrics["test_correct_l2_pct"]),
            "wrong_l2_correct_l1_pct": float(test_metrics["test_wrong_l2_correct_l1_pct"]),
            "wrong_l1_pct": float(test_metrics["test_wrong_l1_pct"]),
            "mean_hier_distance": float(test_metrics["test_mean_hier_distance"]),
            "mean_hier_distance_norm": float(test_metrics["test_mean_hier_distance_norm"]),
            "HP": float(test_metrics["test_HP"]),
            "HR": float(test_metrics["test_HR"]),
            "HF1": float(test_metrics["test_HF1"]),
        }
        hier_error_path = os.path.join(decoder_dir, "hierarchical_error_decomposition_test.csv")
        pd.DataFrame([hier_error]).to_csv(hier_error_path, index=False)

        summary = {
            "experiment_name": cfg["experiment_name"],
            "decoder": decoder_cfg.get("decoder", "base_l2"),
            "decoder_name": decoder_name,
            "decoder_top_k": decoder_cfg.get("top_k"),
            "decoder_alpha": decoder_cfg.get("alpha"),
            "seed": SEED,
            "fold": test_fold,
            "val_fold": val_fold,
            "model_name": MODEL_NAME,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "wd": WD,
            "max_len": MAX_LEN,
            "max_epochs": MAX_EPOCHS,
            "focal_gamma": FOCAL_GAMMA,
            "class_weight_clip": MAX_CLASS_WEIGHT_CLIP,
            "lambda_l1": cfg.get("lambda_l1", 0.05),
            "lambda_hgclr": cfg.get("lambda_hgclr", 0.05),
            "contrast_temp": cfg.get("contrast_temp", 0.10),
            "same_l2_weight": cfg.get("same_l2_weight", 1.0),
            "sibling_negative_weight": cfg.get("sibling_negative_weight", 1.25),
            "projection_dim": cfg.get("projection_dim", 256),
            "train_size": int(len(train_idx)),
            "val_size": int(len(val_idx)),
            "test_size": int(len(test_idx)),
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "prediction_file": pred_path,
            "classification_report_file": report_path,
            "per_class_file": per_class_path,
            "per_parent_file": per_parent_path,
            "hier_error_file": hier_error_path,
            "loss_log_file": loss_log_path,
            "loss_plot_file": loss_plot_path,
        }

        # Keep base Trainer eval metrics for traceability but report decoder-specific val_* metrics separately.
        for k, v in eval_metrics_from_trainer.items():
            summary[f"trainer_{k}"] = v

        summary.update(cm_paths)
        summary.update(val_metrics)
        summary.update(test_metrics)

        summary_path = os.path.join(decoder_dir, "run_summary_hgclr_cv.json")
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)

        print(f"\nRESULT [{decoder_name}]:")
        print("  val macro-F1:", summary.get("val_macro_f1"))
        print("  test macro-F1:", summary.get("test_macro_f1"))
        print("  test HF1:", summary.get("test_HF1"))
        print("  test wrong-L1 pct:", summary.get("test_wrong_l1_pct"))

        summary_rows.append(summary)
        pred_dfs.append(pred_df)
        per_class_dfs.append(per_class_df)
        per_parent_dfs.append(per_parent_df)
        hier_error_dfs.append(pd.DataFrame([hier_error]))

    print("  best ckpt:", trainer.state.best_model_checkpoint)

    # Clean GPU memory.
    del trainer, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        summary_rows,
        pd.concat(pred_dfs, ignore_index=True),
        pd.concat(per_class_dfs, ignore_index=True),
        pd.concat(per_parent_dfs, ignore_index=True),
        pd.concat(hier_error_dfs, ignore_index=True),
    )


In [ ]:

# ============================================================
# 15) Aggregate CV report: fold mean/std and OOF summaries by decoder
# ============================================================

def mean_std_str(vals):
    vals = pd.Series(vals).dropna().astype(float)
    if len(vals) == 0:
        return ""
    mean = vals.mean()
    std = vals.std(ddof=1) if len(vals) > 1 else 0.0
    return f"{mean:.4f} ± {std:.4f}"


def aggregate_fold_results(df, group_cols=("experiment_name", "decoder_name")):
    ignore = set(group_cols) | {
        "seed", "fold", "val_fold", "best_checkpoint", "prediction_file",
        "classification_report_file", "per_class_file", "per_parent_file",
        "hier_error_file", "loss_log_file", "loss_plot_file", "model_name"
    }
    metric_cols = [
        c for c in df.columns
        if c not in ignore and pd.api.types.is_numeric_dtype(df[c])
    ]

    rows = []
    for keys, g in df.groupby(list(group_cols)):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: val for col, val in zip(group_cols, keys)}
        row["seed"] = SEED
        row["n_folds"] = g["fold"].nunique()
        row["folds"] = ",".join(map(str, sorted(g["fold"].unique())))

        for meta in [
            "decoder", "decoder_top_k", "decoder_alpha",
            "lambda_l1", "lambda_hgclr", "contrast_temp", "same_l2_weight",
            "sibling_negative_weight", "projection_dim", "focal_gamma",
            "class_weight_clip", "lr", "batch_size", "max_epochs", "encoder_dropout",
            "attention_dropout", "classifier_dropout", "projection_dropout",
        ]:
            if meta in g.columns:
                row[meta] = g[meta].iloc[0]

        for m in metric_cols:
            vals = g[m].dropna().astype(float)
            if len(vals) > 0:
                row[f"{m}_mean"] = float(vals.mean())
                row[f"{m}_std"] = float(vals.std(ddof=1) if len(vals) > 1 else 0.0)
                row[f"{m}_mean±std"] = mean_std_str(vals)
        rows.append(row)
    return pd.DataFrame(rows)


def aggregate_per_class(per_class_df):
    group_cols = ["experiment_name", "decoder_name", "decoder", "L1", "L2", "l2_id"]
    metric_cols = ["precision", "recall", "f1", "support"]
    rows = []
    for keys, g in per_class_df.groupby(group_cols):
        row = {col: val for col, val in zip(group_cols, keys)}
        row["seed"] = SEED
        row["n_folds"] = g["fold"].nunique()
        if "decoder_top_k" in g.columns:
            row["decoder_top_k"] = g["decoder_top_k"].iloc[0]
        if "decoder_alpha" in g.columns:
            row["decoder_alpha"] = g["decoder_alpha"].iloc[0]
        for m in metric_cols:
            vals = g[m].dropna().astype(float)
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_std"] = float(vals.std(ddof=1) if len(vals) > 1 else 0.0)
            row[f"{m}_mean±std"] = mean_std_str(vals)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["experiment_name", "decoder_name", "L1", "f1_mean", "L2"], ascending=[True, True, True, True, True])


def aggregate_per_parent(per_parent_df):
    group_cols = ["experiment_name", "decoder_name", "decoder", "L1", "l1_id"]
    metric_cols = [
        "support", "l2_accuracy_within_true_parent", "l2_macro_f1_within_true_parent",
        "parent_from_l2_accuracy", "wrong_parent_rate", "wrong_l2_but_correct_parent_rate",
        "l1_aux_accuracy", "aux_parent_child_consistency",
    ]
    rows = []
    for keys, g in per_parent_df.groupby(group_cols):
        row = {col: val for col, val in zip(group_cols, keys)}
        row["seed"] = SEED
        row["n_folds"] = g["fold"].nunique()
        row["n_child_l2"] = int(g["n_child_l2_train"].iloc[0]) if "n_child_l2_train" in g.columns else np.nan
        if "decoder_top_k" in g.columns:
            row["decoder_top_k"] = g["decoder_top_k"].iloc[0]
        if "decoder_alpha" in g.columns:
            row["decoder_alpha"] = g["decoder_alpha"].iloc[0]
        for m in metric_cols:
            if m in g.columns:
                vals = g[m].dropna().astype(float)
                row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
                row[f"{m}_std"] = float(vals.std(ddof=1) if len(vals) > 1 else 0.0)
                row[f"{m}_mean±std"] = mean_std_str(vals)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["experiment_name", "decoder_name", "l2_macro_f1_within_true_parent_mean", "L1"])


def aggregate_hier_error(hier_error_df):
    group_cols = ["experiment_name", "decoder_name", "decoder"]
    metric_cols = [
        "n", "correct_l2_count", "wrong_l2_correct_l1_count", "wrong_l1_count",
        "correct_l2_pct", "wrong_l2_correct_l1_pct", "wrong_l1_pct",
        "mean_hier_distance", "mean_hier_distance_norm", "HP", "HR", "HF1",
    ]

    rows = []
    for keys, g in hier_error_df.groupby(group_cols):
        row = {col: val for col, val in zip(group_cols, keys)}
        row["seed"] = SEED
        row["n_folds"] = g["fold"].nunique()
        if "decoder_top_k" in g.columns:
            row["decoder_top_k"] = g["decoder_top_k"].iloc[0]
        if "decoder_alpha" in g.columns:
            row["decoder_alpha"] = g["decoder_alpha"].iloc[0]
        for m in metric_cols:
            vals = g[m].dropna().astype(float)
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_std"] = float(vals.std(ddof=1) if len(vals) > 1 else 0.0)
            row[f"{m}_mean±std"] = mean_std_str(vals)
        rows.append(row)
    return pd.DataFrame(rows)


def compute_oof_metrics_and_plots(oof_pred_df, exp_name, decoder_name, out_dir):
    y_l2 = oof_pred_df["true_l2_id"].astype(int).values
    pred_l2 = oof_pred_df["pred_l2_id"].astype(int).values
    y_l1 = oof_pred_df["true_l1_id"].astype(int).values
    pred_l1_aux = oof_pred_df["pred_l1_aux_id"].astype(int).values if "pred_l1_aux_id" in oof_pred_df.columns else None

    metrics = {
        "experiment_name": exp_name,
        "decoder_name": decoder_name,
        "decoder": oof_pred_df["decoder"].iloc[0] if "decoder" in oof_pred_df.columns else decoder_name,
        "decoder_top_k": oof_pred_df["decoder_top_k"].iloc[0] if "decoder_top_k" in oof_pred_df.columns else None,
        "decoder_alpha": oof_pred_df["decoder_alpha"].iloc[0] if "decoder_alpha" in oof_pred_df.columns else None,
        "seed": SEED,
        "n_oof": len(oof_pred_df),
    }
    metrics.update(compute_overall_metrics(y_l2, pred_l2, y_l1, pred_l1_aux=pred_l1_aux, prefix="oof_"))

    safe_name = f"{exp_name}_{decoder_name}"

    # OOF L2 confusion matrix.
    cm_l2 = confusion_matrix(y_l2, pred_l2, labels=list(range(num_l2)))
    cm_l2_df = pd.DataFrame(
        cm_l2,
        index=[f"true_{l2_id2lab[i]}" for i in range(num_l2)],
        columns=[f"pred_{l2_id2lab[i]}" for i in range(num_l2)],
    )
    cm_l2_csv = os.path.join(out_dir, f"{safe_name}_OOF_confusion_matrix_L2.csv")
    cm_l2_png = os.path.join(out_dir, f"{safe_name}_OOF_confusion_matrix_L2.png")
    cm_l2_df.to_csv(cm_l2_csv)
    save_cm_plot(
        cm_l2,
        [f"true_{l2_id2lab[i]}" for i in range(num_l2)],
        [f"pred_{l2_id2lab[i]}" for i in range(num_l2)],
        cm_l2_png,
        f"{safe_name} OOF L2 confusion matrix",
        figsize=(18, 16),
    )

    # OOF parent confusion matrix derived from predicted L2.
    pred_l1_from_l2 = parent_id_for_l2[pred_l2]
    cm_l1 = confusion_matrix(y_l1, pred_l1_from_l2, labels=list(range(num_l1)))
    cm_l1_df = pd.DataFrame(
        cm_l1,
        index=[f"true_{l1_id2lab[i]}" for i in range(num_l1)],
        columns=[f"pred_{l1_id2lab[i]}" for i in range(num_l1)],
    )
    cm_l1_csv = os.path.join(out_dir, f"{safe_name}_OOF_confusion_matrix_L1_from_L2.csv")
    cm_l1_png = os.path.join(out_dir, f"{safe_name}_OOF_confusion_matrix_L1_from_L2.png")
    cm_l1_df.to_csv(cm_l1_csv)
    save_cm_plot(
        cm_l1,
        [f"true_{l1_id2lab[i]}" for i in range(num_l1)],
        [f"pred_{l1_id2lab[i]}" for i in range(num_l1)],
        cm_l1_png,
        f"{safe_name} OOF parent confusion matrix from L2",
        figsize=(8, 7),
    )

    # OOF hierarchical error bar plot.
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ["Correct L2", "Wrong L2 / correct L1", "Wrong L1"]
    vals = [
        metrics["oof_correct_l2_pct"],
        metrics["oof_wrong_l2_correct_l1_pct"],
        metrics["oof_wrong_l1_pct"],
    ]
    ax.bar(bars, vals)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Rate")
    ax.set_title(f"{safe_name} OOF hierarchical error decomposition")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    err_png = os.path.join(out_dir, f"{safe_name}_OOF_error_decomposition.png")
    fig.savefig(err_png, dpi=200, bbox_inches="tight")
    plt.close(fig)

    metrics.update({
        "oof_cm_l2_csv": cm_l2_csv,
        "oof_cm_l2_png": cm_l2_png,
        "oof_cm_l1_from_l2_csv": cm_l1_csv,
        "oof_cm_l1_from_l2_png": cm_l1_png,
        "oof_error_decomposition_png": err_png,
    })

    return metrics


def plot_all_fold_losses(summary_df, exp_name, out_dir):
    fig, ax = plt.subplots(figsize=(10, 6))
    any_line = False

    # Loss is identical across decoders for the same trained fold, so drop duplicate decoder rows.
    exp_rows = summary_df[summary_df["experiment_name"] == exp_name].drop_duplicates(subset=["fold", "loss_log_file"])

    for _, row in exp_rows.iterrows():
        p = row.get("loss_log_file")
        fold = row.get("fold")
        if isinstance(p, str) and os.path.exists(p):
            log_df = pd.read_csv(p)
            if "epoch" in log_df.columns and "eval_loss" in log_df.columns:
                ev = log_df.dropna(subset=["eval_loss"])
                if not ev.empty:
                    ax.plot(ev["epoch"], ev["eval_loss"], marker="o", label=f"fold {fold} eval")
                    any_line = True
            if "epoch" in log_df.columns and "loss" in log_df.columns:
                tr = log_df.dropna(subset=["loss"])
                if not tr.empty:
                    ax.plot(tr["epoch"], tr["loss"], linestyle="--", alpha=0.5, label=f"fold {fold} train")
                    any_line = True

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"{exp_name}: loss across CV folds")
    ax.grid(True, alpha=0.3)
    if any_line:
        ax.legend(fontsize=8, ncol=2)

    fig.tight_layout()
    plot_path = os.path.join(out_dir, f"{exp_name}_all_folds_loss_overlay.png")
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return plot_path


In [ ]:

# ============================================================
# 16) Run all five folds and save the consolidated report
# ============================================================

if RUN_HGCLR_CV:
    all_summaries = []
    all_oof_preds = []
    all_per_class = []
    all_per_parent = []
    all_hier_error = []

    for cfg in HGCLR_CONFIGS:
        for test_fold in range(N_FOLDS):
            summary_rows, pred_df, pc_df, pp_df, he_df = run_one_cv_fold_hgclr(cfg, test_fold)

            all_summaries.extend(summary_rows)
            all_oof_preds.append(pred_df)
            all_per_class.append(pc_df)
            all_per_parent.append(pp_df)
            all_hier_error.append(he_df)

            # Save partial outputs after each fold.
            pd.DataFrame(all_summaries).to_csv(os.path.join(OUTPUT_DIR, "PARTIAL_HGCLR_CV_fold_results.csv"), index=False)
            pd.concat(all_oof_preds, ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, "PARTIAL_HGCLR_CV_oof_predictions.csv"), index=False)
            pd.concat(all_per_class, ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, "PARTIAL_HGCLR_CV_per_class.csv"), index=False)
            pd.concat(all_per_parent, ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, "PARTIAL_HGCLR_CV_per_parent.csv"), index=False)
            pd.concat(all_hier_error, ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, "PARTIAL_HGCLR_CV_hier_error.csv"), index=False)

    summary_df = pd.DataFrame(all_summaries)
    oof_pred_df = pd.concat(all_oof_preds, ignore_index=True)
    per_class_fold_df = pd.concat(all_per_class, ignore_index=True)
    per_parent_fold_df = pd.concat(all_per_parent, ignore_index=True)
    hier_error_fold_df = pd.concat(all_hier_error, ignore_index=True)

    summary_agg_df = aggregate_fold_results(summary_df)
    per_class_agg_df = aggregate_per_class(per_class_fold_df)
    per_parent_agg_df = aggregate_per_parent(per_parent_fold_df)
    hier_error_agg_df = aggregate_hier_error(hier_error_fold_df)

    oof_metrics_rows = []
    for (exp_name, decoder_name), g in oof_pred_df.groupby(["experiment_name", "decoder_name"]):
        exp_out_dir = os.path.join(OUTPUT_DIR, f"OOF_ANALYSIS_{exp_name}_{decoder_name}")
        os.makedirs(exp_out_dir, exist_ok=True)
        oof_metrics_rows.append(compute_oof_metrics_and_plots(g, exp_name, decoder_name, exp_out_dir))

    # One loss overlay per trained experiment, because decoders are inference-only.
    for exp_name in sorted(summary_df["experiment_name"].unique()):
        exp_out_dir = os.path.join(OUTPUT_DIR, f"OOF_ANALYSIS_{exp_name}_loss")
        os.makedirs(exp_out_dir, exist_ok=True)
        plot_all_fold_losses(summary_df, exp_name, exp_out_dir)

    oof_metrics_df = pd.DataFrame(oof_metrics_rows)

    # Decoder comparison sorted by the main test and hierarchical metrics.
    decoder_comparison_df = summary_agg_df.copy()
    sort_cols = [c for c in ["test_macro_f1_mean", "test_HF1_mean", "test_parent_from_l2_acc_mean"] if c in decoder_comparison_df.columns]
    if sort_cols:
        decoder_comparison_df = decoder_comparison_df.sort_values(sort_cols, ascending=[False] * len(sort_cols))

    paths = {
        "fold_results": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_fold_results_all_decoders.csv"),
        "mean_std_over_folds": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_summary_mean_std_over_folds_all_decoders.csv"),
        "decoder_comparison": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_decoder_comparison.csv"),
        "oof_predictions": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_oof_predictions_all_decoders.csv"),
        "oof_metrics": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_oof_metrics_all_decoders.csv"),
        "per_class_fold": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_per_class_fold_all_decoders.csv"),
        "per_class_mean_std": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_per_class_mean_std_all_decoders.csv"),
        "per_parent_fold": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_per_parent_fold_all_decoders.csv"),
        "per_parent_mean_std": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_per_parent_mean_std_all_decoders.csv"),
        "hier_error_fold": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_hier_error_fold_all_decoders.csv"),
        "hier_error_mean_std": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_hier_error_mean_std_all_decoders.csv"),
        "excel_report": os.path.join(OUTPUT_DIR, "FINAL_HGCLR_CV_full_report_all_decoders.xlsx"),
    }

    summary_df.to_csv(paths["fold_results"], index=False)
    summary_agg_df.to_csv(paths["mean_std_over_folds"], index=False)
    decoder_comparison_df.to_csv(paths["decoder_comparison"], index=False)
    oof_pred_df.to_csv(paths["oof_predictions"], index=False)
    oof_metrics_df.to_csv(paths["oof_metrics"], index=False)
    per_class_fold_df.to_csv(paths["per_class_fold"], index=False)
    per_class_agg_df.to_csv(paths["per_class_mean_std"], index=False)
    per_parent_fold_df.to_csv(paths["per_parent_fold"], index=False)
    per_parent_agg_df.to_csv(paths["per_parent_mean_std"], index=False)
    hier_error_fold_df.to_csv(paths["hier_error_fold"], index=False)
    hier_error_agg_df.to_csv(paths["hier_error_mean_std"], index=False)

    with pd.ExcelWriter(paths["excel_report"], engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="fold_results", index=False)
        summary_agg_df.to_excel(writer, sheet_name="mean_std_over_folds", index=False)
        decoder_comparison_df.to_excel(writer, sheet_name="decoder_comparison", index=False)
        oof_metrics_df.to_excel(writer, sheet_name="oof_metrics", index=False)
        hier_error_fold_df.to_excel(writer, sheet_name="hier_error_fold", index=False)
        hier_error_agg_df.to_excel(writer, sheet_name="hier_error_mean_std", index=False)
        per_parent_fold_df.to_excel(writer, sheet_name="per_parent_fold", index=False)
        per_parent_agg_df.to_excel(writer, sheet_name="per_parent_mean_std", index=False)
        per_class_fold_df.to_excel(writer, sheet_name="per_class_fold", index=False)
        per_class_agg_df.to_excel(writer, sheet_name="per_class_mean_std", index=False)
        oof_pred_df.to_excel(writer, sheet_name="oof_predictions", index=False)

    print("\nSaved final outputs:")
    for k, v in paths.items():
        print(f"  {k}: {v}")

    print("\nOverall mean ± std over folds by decoder:")
    display(summary_agg_df)

    print("\nDecoder comparison:")
    display(decoder_comparison_df)

    print("\nOOF metrics by decoder:")
    display(oof_metrics_df)

    print("\nWorst L2 classes by mean F1:")
    cols = ["experiment_name", "decoder_name", "L1", "L2", "support_mean±std", "precision_mean±std", "recall_mean±std", "f1_mean±std"]
    display(per_class_agg_df[[c for c in cols if c in per_class_agg_df.columns]].head(30))

    print("\nWorst parents by within-parent L2 macro-F1:")
    cols = ["experiment_name", "decoder_name", "L1", "support_mean±std", "n_child_l2", "l2_macro_f1_within_true_parent_mean±std", "wrong_parent_rate_mean±std", "wrong_l2_but_correct_parent_rate_mean±std"]
    display(per_parent_agg_df[[c for c in cols if c in per_parent_agg_df.columns]])

    print("\nHierarchical error decomposition by decoder:")
    display(hier_error_agg_df)
else:
    print("RUN_HGCLR_CV is False. Set it to True to run experiments.")



RUN: HGCLR_l1_005_contrast_01_t010_sibneg125 | seed=42 | test_fold=0
Train/Val/Test sizes: 1075 359 359
Val fold: 1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

,experiment_name,decoder_name,L1,L2,support_mean±std,precision_mean±std,recall_mean±std,f1_mean±std
0,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,compatibility,interoperability,19.4000 ± 0.5477,0.7269 ± 0.0682,0.6089 ± 0.1172,0.6599 ± 0.0888
2,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,maintainability,modularity,4.6000 ± 0.5477,0.6400 ± 0.2074,0.6100 ± 0.0894,0.6124 ± 0.0989
1,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,maintainability,modifiability,13.2000 ± 0.4472,0.6944 ± 0.1304,0.6374 ± 0.0936,0.6575 ± 0.0735
3,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,maintainability,testability,5.4000 ± 0.5477,0.8533 ± 0.0837,0.7000 ± 0.3037,0.7206 ± 0.2261
5,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,performanceefficiency,resourceutilization,11.6000 ± 0.5477,0.6393 ± 0.1107,0.8273 ± 0.0876,0.7162 ± 0.0875
4,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,performanceefficiency,capacity,16.0000 ± 0.0000,0.8663 ± 0.0940,0.8000 ± 0.1027,0.8246 ± 0.0557
7,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,performanceefficiency,timebehaviour,37.0000 ± 0.0000,0.8329 ± 0.0405,0.8378 ± 0.0917,0.8316 ± 0.0354
6,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,performanceefficiency,scalability,6.0000 ± 0.0000,0.9429 ± 0.1278,0.8333 ± 0.2041,0.8690 ± 0.1473
9,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,portability,installability,3.8000 ± 0.4472,0.4667 ± 0.1394,0.4167 ± 0.1179,0.4381 ± 0.1221
8,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,portability,adaptability,21.8000 ± 0.4472,0.7320 ± 0.1236,0.7874 ± 0.1111,0.7516 ± 0.0827



Worst parents by within-parent L2 macro-F1:


,experiment_name,decoder_name,L1,support_mean±std,n_child_l2,l2_macro_f1_within_true_parent_mean±std,wrong_parent_rate_mean±std,wrong_l2_but_correct_parent_rate_mean±std
6,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,usability,70.8000 ± 0.4472,6,0.6459 ± 0.0799,0.1326 ± 0.0460,0.2005 ± 0.0556
5,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,security,90.0000 ± 1.2247,4,0.6866 ± 0.0187,0.0533 ± 0.0087,0.2224 ± 0.0225
3,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,portability,25.6000 ± 0.8944,2,0.6945 ± 0.0912,0.2058 ± 0.1070,0.0622 ± 0.0202
1,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,maintainability,23.2000 ± 0.4472,3,0.7240 ± 0.0621,0.2591 ± 0.0826,0.0946 ± 0.0351
4,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,reliability,59.0000 ± 1.0000,4,0.7374 ± 0.0939,0.0949 ± 0.0364,0.1498 ± 0.0609
0,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,compatibility,19.4000 ± 0.5477,1,0.7518 ± 0.0883,0.3911 ± 0.1172,0.0000 ± 0.0000
2,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,performanceefficiency,70.6000 ± 0.5477,4,0.8444 ± 0.0352,0.0707 ± 0.0421,0.1019 ± 0.0227
13,HGCLR_l1_005_contrast_01_t010_sibneg125,hard_mask,usability,70.8000 ± 0.4472,6,0.6459 ± 0.0799,0.1326 ± 0.0460,0.2005 ± 0.0556
12,HGCLR_l1_005_contrast_01_t010_sibneg125,hard_mask,security,90.0000 ± 1.2247,4,0.6866 ± 0.0187,0.0533 ± 0.0087,0.2224 ± 0.0225
10,HGCLR_l1_005_contrast_01_t010_sibneg125,hard_mask,portability,25.6000 ± 0.8944,2,0.6945 ± 0.0912,0.2058 ± 0.1070,0.0622 ± 0.0202



Hierarchical error decomposition by decoder:


,experiment_name,decoder_name,decoder,seed,n_folds,decoder_top_k,decoder_alpha,n_mean,n_std,n_mean±std,...,mean_hier_distance_norm_mean±std,HP_mean,HP_std,HP_mean±std,HR_mean,HR_std,HR_mean±std,HF1_mean,HF1_std,HF1_mean±std
0,HGCLR_l1_005_contrast_01_t010_sibneg125,base_l2,base_l2,42,5,NaN,NaN,358.6,0.547723,358.6000 ± 0.5477,...,0.1969 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229
1,HGCLR_l1_005_contrast_01_t010_sibneg125,hard_mask,hard_mask,42,5,1.0,NaN,358.6,0.547723,358.6000 ± 0.5477,...,0.1969 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229
2,HGCLR_l1_005_contrast_01_t010_sibneg125,soft_fusion,soft_fusion,42,5,NaN,1.0,358.6,0.547723,358.6000 ± 0.5477,...,0.1969 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229
3,HGCLR_l1_005_contrast_01_t010_sibneg125,topk_parent_2,topk_parent,42,5,2.0,NaN,358.6,0.547723,358.6000 ± 0.5477,...,0.1969 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229
4,HGCLR_l1_005_contrast_01_t010_sibneg125,topk_parent_3,topk_parent,42,5,3.0,NaN,358.6,0.547723,358.6000 ± 0.5477,...,0.1969 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229,0.803101,0.022948,0.8031 ± 0.0229
